In [1]:
# imports
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split

In [24]:
# load clean data (no pickle needed)
X = np.load("../data/processed/X.npy")
y = np.load("../data/processed/y.npy")

In [25]:
# check dtype and shape
X.dtype, y.dtype, X.shape, y.shape

(dtype('float32'), dtype('int64'), (2193156, 75), (2193156,))

In [4]:
# use a subset (for faster training)
X_small = X[:200000]
y_small = y[:200000]

In [10]:
# convert to numeric (remove commas)
X_small = np.array([
    [float(str(value).replace(',', '')) for value in row]
    for row in X_small
], dtype=np.float32)

In [11]:
# split again
X_train, X_test, y_train, y_test = train_test_split(
    X_small, y_small, test_size=0.2, random_state=42
)

In [13]:
# normalization
mean = X_train.mean(axis=0)
std = X_train.std(axis=0)

std[std == 0] = 1

X_train = (X_train - mean) / std
X_test = (X_test - mean) / std

In [14]:
# convert to tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

In [15]:
# check shapes
X_train.shape, y_train.shape

(torch.Size([160000, 75]), torch.Size([160000]))

In [16]:
# define model
class LaneChangeModel(nn.Module):
    def __init__(self):
        super(LaneChangeModel, self).__init__()
        
        self.model = nn.Sequential(
            nn.Linear(75, 64),   # input → hidden
            nn.ReLU(),
            
            nn.Linear(64, 32),   # hidden → hidden
            nn.ReLU(),
            
            nn.Linear(32, 1),    # output
            nn.Sigmoid()
        )
    
    def forward(self, x):
        return self.model(x)

In [17]:
# create model instance
model = LaneChangeModel()

model

LaneChangeModel(
  (model): Sequential(
    (0): Linear(in_features=75, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=32, bias=True)
    (3): ReLU()
    (4): Linear(in_features=32, out_features=1, bias=True)
    (5): Sigmoid()
  )
)

In [18]:
# loss function
criterion = nn.BCELoss()

In [19]:
# optimizer
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [20]:
# training loop
epochs = 5

for epoch in range(epochs):
    
    model.train()
    
    # forward pass
    outputs = model(X_train).squeeze()
    
    # compute loss
    loss = criterion(outputs, y_train)
    
    # backward pass
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}")

Epoch 1/5, Loss: 0.6067
Epoch 2/5, Loss: 0.5954
Epoch 3/5, Loss: 0.5837
Epoch 4/5, Loss: 0.5716
Epoch 5/5, Loss: 0.5593


In [21]:
# evaluation mode
model.eval()

with torch.no_grad():
    outputs = model(X_test).squeeze()

In [22]:
# convert probabilities to predictions
preds = (outputs >= 0.5).float()

In [23]:
# compute accuracy
accuracy = (preds == y_test).float().mean()

accuracy.item()

0.8868749737739563